# Quick start

This notebook loads a small feeder written in `.cimtbl` and reads it with the getters in `cimhub_core.getters`: a line, a transformer, loads, a PV inverter and a terminal. It runs from `docs/guide/` in the repository, next to [`feeder.cimtbl`](https://github.com/PNNL-CIM-Tools/CIMHub/blob/develop/cimhub_core/docs/guide/feeder.cimtbl).

## The model

One source, a 2000 ft line with its own sequence impedance, a 500 kVA transformer to 480 V, three loads and a PV inverter. Units are on the column headers, and CIM holds them as SI.

In [1]:
from pathlib import Path

print(Path('feeder.cimtbl').read_text())

# A small feeder for the quick start: a source, a 4.16 kV to 480 V transformer, a line with its own sequence
# impedance, a PV unit and three loads, one of them on a daily curve. The transformer follows the
# transformer_template shape and the load schedule follows the load_schedule shape, both in cimhub_core/shapes.
# Written for these docs; no external origin.

Profile = cim18v17
Authority = https://github.com/PNNL-CIM-Tools/CIMHub
Model = guide_feeder

Object BasePower: mRID=sbase, basePower (MVA)=10
Object BaseFrequency: mRID=base_freq, frequency (Hz)=60
Object Substation: mRID=sub, name=sub
Object Feeder: mRID=fdr, name=guide_feeder, NormalEnergizingSubstation=sub

Table BaseVoltage: mRID, nominalVoltage (kV)
    bv_4p16kv, 4.16
    bv_480v,   0.48

Table LoadResponseCharacteristic: mRID, exponentModel, pConstantPower, pConstantCurrent, pConstantImpedance, qConstantPower, qConstantCurrent, qConstantImpedance
    zip_pq, false, 1, 0, 0, 1, 0, 0
    zip_z,  false, 0, 0, 1, 0, 0, 1



## Load it

`cimtables.load_model` builds a cim-graph `FeederModel`. It also sets the CIM profile for the whole process, so the load runs inside `cim_scope()`, which puts the profile back afterwards. The getters work on the loaded objects either way.

In [2]:
import cimprofiles.cim18.cim18v17 as cim
import cimtables

from cimhub_core.cimtbl_testing import cim_scope

with cim_scope():
    network = cimtables.load_model('feeder.cimtbl')

print(type(network).__name__, network.base_power.to('MVA'))

FeederModel 10.0


A getter takes a CIM object, so a small helper finds one by mRID.

In [3]:
def find(cim_class, mrid):
    return next(obj for obj in network.graph[cim_class].values() if obj.mRID == mrid)


line = find(cim.ACLineSegment, 'ln_trunk')
transformer = find(cim.PowerTransformer, 'xf1')
print(line.mRID, transformer.mRID)

ln_trunk xf1


## A line

`get_line_sequence_impedance` reads the whole line's values. This line has no lumped `r` and `x` of its own: they come from its `PerLengthSequenceImpedance`. By default the getter looks only at the lumped values and says so. `derive=` lets it go one step further.

In [4]:
from cimhub_core.getters.line_impedance import ImpedanceSource, get_line_sequence_impedance

try:
    get_line_sequence_impedance(line)
except ValueError as error:
    print(error)

ACLineSegment 'ln_trunk': no sequence impedance within derive=LUMPED (lumped r/x: unset, PerLengthImpedance: PerLengthSequenceImpedance, geometry: none)


In [5]:
z = get_line_sequence_impedance(line, 'ohm', derive=ImpedanceSource.PER_LENGTH_SEQ)
print(z)
print(z.r, z.source.name)

SequenceImpedance(r=0.21335999999999997, x=0.26212799999999997, b=2.37744e-06, g=None, r0=0.359664, x0=0.79248, b0=1.03632e-06, g0=None, source=<ImpedanceSource.PER_LENGTH_SEQ: 2>)
0.21335999999999997 PER_LENGTH_SEQ


The same line in per-unit is on its `BaseVoltage` and the model's base power. Admittance is in siemens. `g` is `None` because the line has no conductance, and `None` is not zero: it says CIM leaves the value unset.

In [6]:
z_pu = get_line_sequence_impedance(line, 'pu', base_power=network.base_power, derive=ImpedanceSource.PER_LENGTH_SEQ)
print(round(z_pu.r, 4), round(z_pu.x, 4), z_pu.g)

0.1233 0.1515 None


The per-length values come in the units you name, here ohm/km and microsiemens/km. They are the numbers the file was written with.

In [7]:
from cimhub_core.getters.line_impedance import get_line_code, get_line_sequence_impedance_per_length

per_length = get_line_sequence_impedance_per_length(line, 'ohm/km', 'uS/km')
print(per_length.r, per_length.x, per_length.b)
print(get_line_code(line).mRID)

0.35 0.43 3.900000000000001
ln_trunk_lc


## A transformer

A `PowerTransformer` is read through its units: its tanks, or the transformer itself when it has none. A three-phase transformer here has one tank, and a bank has one per can. `get_transformer_model` returns the windings and the mesh impedance. The ratings are in the units you name, and the impedance is percent on the power base it reports.

In [8]:
from cimhub_core.getters.transformer import get_transformer_model, get_transformer_units

[unit] = get_transformer_units(transformer)
model = get_transformer_model(unit, 'percent', voltage_unit='kV', power_unit='kVA')
print(model.form.name, model.base_power)
for winding in model.windings:
    print(winding.number, winding.connection.value, winding.rated_u, winding.rated_s, winding.grounded)
print(model.mesh)

BY_TANK_ENDS 500.0
1 Yn 4.16 500.0 True
2 Yn 0.48 500.0 True
{(1, 2): MeshImpedance(r=1.1, x=2.0)}


The same impedance on the model's 10 MVA base, as per-unit.

In [9]:
on_system = get_transformer_model(unit, 'pu', base_power=network.base_power, voltage_unit='kV', power_unit='kVA')
print(on_system.base_power, on_system.mesh[(1, 2)])

10000.0 MeshImpedance(r=0.22, x=0.4)


## Loads

`get_phase_loads` returns the total and each phase's share. `split=True` means the loader split the total evenly; `False` means each phase has its own values.

In [10]:
from cimhub_core.getters.load import get_phase_loads

balanced = get_phase_loads(find(cim.EnergyConsumer, 'ld_n1'), 'kW', 'kVAr')
print(balanced.total, balanced.split)
print({phase.value: power for phase, power in balanced.per_phase.items()})

PhasePower(p=1155.0, q=660.0) True
{'A': PhasePower(p=385.0, q=220.0), 'B': PhasePower(p=385.0, q=220.0), 'C': PhasePower(p=385.0, q=220.0)}


In [11]:
unbalanced = get_phase_loads(find(cim.EnergyConsumer, 'ld_n2'), 'kW', 'kVAr')
print(unbalanced.total, unbalanced.split)
print({phase.value: power for phase, power in unbalanced.per_phase.items()})

PhasePower(p=400.0, q=290.0) False
{'A': PhasePower(p=160.0, q=110.0), 'B': PhasePower(p=120.0, q=90.0), 'C': PhasePower(p=120.0, q=90.0)}


Without unit arguments the getter returns SI, W and VAr. Name the unit.

In [12]:
print(get_phase_loads(find(cim.EnergyConsumer, 'ld_n1')).total)

PhasePower(p=1155000.0, q=660000.0)


## Generation and sign

CIM's sign is the load's: power into the equipment is positive, so a PV inverter's `p` is negative. `get_der_output` makes the caller choose the sign it wants.

In [13]:
from cimhub_core.getters.der import get_der_output

inverter = find(cim.PowerElectronicsConnection, 'pv_p1')
print(get_der_output(inverter, 'kW', 'kVAr', sign='load').total)
print(get_der_output(inverter, 'kW', 'kVAr', sign='generation').total)

PhasePower(p=-240.0, q=0.0)
PhasePower(p=240.0, q=0.0)


## Terminals and phases

A transformer's two ends have terminals on different nodes. `get_terminal_phases` reads one terminal's conductors, in A, B, C, s1, s2, N order.

In [14]:
from cimhub_core.getters.terminals import get_conductor_phases, get_terminal_nodes, get_terminal_phases

print([node.mRID for node in get_terminal_nodes(line)])
print([phase.value for phase in get_conductor_phases(line)])
for terminal in sorted(transformer.Terminals, key=lambda t: t.sequenceNumber):
    print(terminal.sequenceNumber, [phase.value for phase in get_terminal_phases(terminal)])

['sourcebus', 'n1']
['A', 'B', 'C']
1 ['A', 'B', 'C', 'N']
2 ['A', 'B', 'C', 'N']


## Unset values and decisions

A getter that needs a value CIM leaves unset raises, and one that CIM may legitimately leave unset returns `None`. A getter that depends on a choice, such as which state of a switch, makes you pass it.

In [15]:
from cimhub_core.getters.in_service import get_in_service

print(get_in_service(line, which='actual'))
try:
    get_in_service(line, which='now')
except ValueError as error:
    print(error)

None
which must be one of ['actual', 'normal'], not 'now'


Next: [Using the getters](getters.md) and [Units, bases and signs](units.md).